# Slide Skill Optimizer — Training Notebook

Fine-tunes a small open model (Qwen2.5-7B) to replace Claude Opus 4.6 as the
**optimizer** in the Slide Skill OpenEnv loop.

**Pipeline:**
1. **Data collection** — Run N episodes against the live OpenEnv server using
   Claude Opus 4.6 as the oracle optimizer. Record every (prompt, DESIGN_RULES rewrite, reward) tuple.
2. **Filtering** — Keep only steps where `reward > 0` (the rewrite improved the score).
3. **SFT training** — Fine-tune Qwen2.5-7B-Instruct with Unsloth + TRL SFTTrainer.
4. **Save / push** — Export QLoRA weights; optionally push to HuggingFace Hub.

**Models in the loop:**
- Generator: Claude Sonnet 4.6 (writes pptxgenjs JS)
- Evaluator: Gemini 3.1 Pro (scores the slide with vision)
- Oracle optimizer (data collection): Claude Opus 4.6
- Trained optimizer (inference): Qwen2.5-7B fine-tuned here

**Runtime:** T4 GPU (free tier) is sufficient. A100 is faster.
**Estimated collection time:** ~90s per step × 7 steps × N episodes.

## 1. Install Dependencies

In [ ]:
# Unsloth must be installed before transformers to patch correctly.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q anthropic httpx trl datasets transformers accelerate bitsandbytes
print('Done.')

## 2. Configuration

In [ ]:
import os

# ------------------------------------------------------------------
# API keys
# ------------------------------------------------------------------
# Set these here or via Colab Secrets (Secrets tab in left sidebar)
os.environ.setdefault('ANTHROPIC_API_KEY', '')  # for oracle optimizer (Opus 4.6)

# ------------------------------------------------------------------
# OpenEnv server URL
# ------------------------------------------------------------------
# Option A: HuggingFace Spaces deployment
#   SERVER_URL = 'https://your-username-slide-skill-openenv.hf.space'
# Option B: Local server exposed via ngrok
#   !pip install -q pyngrok
#   from pyngrok import ngrok; ngrok.set_auth_token('YOUR_TOKEN')
#   tunnel = ngrok.connect(8000); SERVER_URL = tunnel.public_url
SERVER_URL = 'https://your-hf-space-url.hf.space'  # <-- EDIT THIS

# ------------------------------------------------------------------
# Data collection
# ------------------------------------------------------------------
N_EPISODES = 20          # Number of full optimization episodes to collect
MAX_STEPS_PER_EPISODE = 7
MIN_REWARD = 0.0         # Only train on steps where reward > this threshold
DATA_FILE = 'trajectories.json'

# ------------------------------------------------------------------
# Training
# ------------------------------------------------------------------
BASE_MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 4096
OUTPUT_DIR = './optimizer-model'
HF_REPO = ''  # Optional: 'your-username/slide-optimizer' to push to Hub

print(f'Server: {SERVER_URL}')
print(f'Collecting {N_EPISODES} episodes, training on steps with reward > {MIN_REWARD}')

## 3. Data Collection

In [ ]:
import json
import textwrap
import time
from dataclasses import dataclass, asdict
from typing import Any

import anthropic
import httpx

In [ ]:
ORACLE_MODEL = 'claude-opus-4-6'
OPTIMIZER_SYSTEM = (
    'You are a McKinsey slide design optimizer. '
    'You improve a PowerPoint generation skill by rewriting its DESIGN_RULES.md file. '
    'Output ONLY the markdown file content — no explanation, no code fences.'
)


def build_optimizer_prompt(obs: dict) -> str:
    """Format an observation dict into the optimizer input prompt."""
    scores = obs['scores']
    strengths = '\n'.join(f'- {s}' for s in obs.get('strengths', []))
    weaknesses = '\n'.join(f'- {w}' for w in obs.get('weaknesses', []))
    return textwrap.dedent(f"""\
        ## Current Score: {obs['total']}/100

        ## Score Breakdown
        - background_layout: {scores['background_layout']}/15
        - color_palette: {scores['color_palette']}/15
        - typography: {scores['typography']}/15
        - title_quality: {scores['title_quality']}/15
        - data_presentation: {scores['data_presentation']}/15
        - structural_elements: {scores['structural_elements']}/15
        - overall_impression: {scores['overall_impression']}/10

        ## Evaluator Feedback
        Strengths:
        {strengths}

        Weaknesses:
        {weaknesses}

        Verdict: {obs['one_line_verdict']}

        ## Current DESIGN_RULES.md
        {obs['design_rules_content']}

        ## Current EXAMPLES.md
        {obs['examples_content']}

        Write an improved DESIGN_RULES.md that addresses the weaknesses above \
        while preserving what works well. Focus on the lowest-scoring dimensions.
    """)


def call_oracle(prompt: str, anthropic_client: anthropic.Anthropic) -> str:
    """Call Claude Opus 4.6 to generate an improved DESIGN_RULES.md."""
    response = anthropic_client.messages.create(
        model=ORACLE_MODEL,
        max_tokens=4096,
        system=OPTIMIZER_SYSTEM,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.content[0].text.strip()


print('Oracle functions defined.')

In [ ]:
@dataclass
class TrainingStep:
    prompt: str       # optimizer input (observation formatted as text)
    completion: str   # oracle output (new DESIGN_RULES.md)
    reward: float     # environment reward for this step
    total: int        # slide score after this step
    episode: int
    step: int


BASELINE_EXAMPLES = '(Empty — no prior optimization rounds)\n'


def collect_episode(
    episode_idx: int,
    http: httpx.Client,
    anthropic_client: anthropic.Anthropic,
    max_steps: int = MAX_STEPS_PER_EPISODE,
) -> list[TrainingStep]:
    """Run one full optimization episode and return training steps."""
    steps = []

    # Reset session.
    resp = http.post(f'{SERVER_URL}/reset', json={})
    resp.raise_for_status()
    session_id = resp.json()['session_id']
    print(f'  Episode {episode_idx}: session={session_id}')

    # Baseline step (no optimizer action — just trigger generation).
    resp = http.post(f'{SERVER_URL}/step', json={
        'session_id': session_id,
        'action': {
            'action_type': 'replace_file',
            'file': 'EXAMPLES.md',
            'new_content': BASELINE_EXAMPLES,
        },
    })
    resp.raise_for_status()
    obs = resp.json()
    print(f'    baseline score={obs["total"]}/100')

    try:
        # Optimization steps.
        for step_idx in range(1, max_steps + 1):
            if obs.get('done'):
                break

            # Build prompt and call oracle.
            prompt = build_optimizer_prompt(obs)
            completion = call_oracle(prompt, anthropic_client)

            # Submit to environment.
            resp = http.post(f'{SERVER_URL}/step', json={
                'session_id': session_id,
                'action': {
                    'action_type': 'replace_file',
                    'file': 'DESIGN_RULES.md',
                    'new_content': completion,
                },
            })
            resp.raise_for_status()
            obs = resp.json()

            steps.append(TrainingStep(
                prompt=prompt,
                completion=completion,
                reward=obs['reward'],
                total=obs['total'],
                episode=episode_idx,
                step=step_idx,
            ))
            delta = f"{obs['reward']*100:+.0f}pts"
            print(f'    step {step_idx}: score={obs["total"]}/100 ({delta})')
    finally:
        # Always clean up session, even on error.
        http.delete(f'{SERVER_URL}/sessions/{session_id}')
    return steps


print('Collection function defined.')

In [ ]:
all_steps: list[TrainingStep] = []

anthropic_client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])

# Long timeout — each step takes 60-120s (LLM + Node + LibreOffice + Gemini).
with httpx.Client(timeout=300.0) as http:
    for ep in range(N_EPISODES):
        print(f'\nEpisode {ep + 1}/{N_EPISODES}')
        try:
            episode_steps = collect_episode(ep + 1, http, anthropic_client)
            all_steps.extend(episode_steps)
        except Exception as e:
            print(f'  Episode {ep + 1} failed: {e} — skipping')
            continue

        # Save incrementally so a Colab crash doesn't lose all data.
        with open(DATA_FILE, 'w') as f:
            json.dump([asdict(s) for s in all_steps], f, indent=2)
        print(f'  Saved {len(all_steps)} steps to {DATA_FILE}')

print(f'\nCollection complete: {len(all_steps)} total steps across {N_EPISODES} episodes.')

In [ ]:
import statistics

# Load from disk (in case of kernel restart).
with open(DATA_FILE) as f:
    raw = json.load(f)
all_steps = [TrainingStep(**s) for s in raw]

rewards = [s.reward for s in all_steps]
positive = [s for s in all_steps if s.reward > MIN_REWARD]

print(f'Total steps collected : {len(all_steps)}')
print(f'Mean reward           : {statistics.mean(rewards):.3f}' if rewards else 'Mean reward           : n/a')
print(f'Positive-reward steps : {len(positive)} ({100*len(positive)/max(len(all_steps),1):.0f}%)')
print(f'Training examples     : {len(positive)}')

if len(positive) < 10:
    print('\n⚠ Very few positive examples. Consider collecting more episodes or lowering MIN_REWARD.')

## 4. Fine-tuning with Unsloth + TRL

In [ ]:
from datasets import Dataset
from unsloth import FastLanguageModel

# Load model first so we can use its tokenizer for chat formatting.
print(f'Loading {BASE_MODEL} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print('Model loaded.')


def format_example(step: TrainingStep) -> dict:
    """Format a training step as a chat-template string for SFT."""
    text = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': OPTIMIZER_SYSTEM},
            {'role': 'user', 'content': step.prompt},
            {'role': 'assistant', 'content': step.completion},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}


formatted = [format_example(s) for s in positive]
dataset = Dataset.from_list(formatted).train_test_split(test_size=0.1, seed=42)

print(f'Train examples: {len(dataset["train"])}')
print(f'Eval  examples: {len(dataset["test"])}')
print('\nSample (truncated):')
print(dataset['train'][0]['text'][:500], '...')

In [ ]:
# Add LoRA adapters via Unsloth.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Unsloth's memory-efficient checkpointing
    random_state=42,
)
print('LoRA adapters attached.')

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,   # effective batch size = 4
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        fp16=not FastLanguageModel.is_bfloat16_supported(),
        bf16=FastLanguageModel.is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none',  # set to 'wandb' if you want W&B logging
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,    # Packs short examples together to fill context — improves throughput
    ),
)

print('Starting training...')
trainer.train()
print('Training complete.')

## 5. Save & Push

In [ ]:
# Save LoRA adapters (small — only the delta weights).
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}/')

# Optional: merge + save full model (larger, but self-contained for deployment).
# model.save_pretrained_merged(OUTPUT_DIR + '-merged', tokenizer, save_method='merged_16bit')

# Optional: push to HuggingFace Hub.
if HF_REPO:
    model.push_to_hub(HF_REPO, token=os.environ.get('HF_TOKEN', ''))
    tokenizer.push_to_hub(HF_REPO, token=os.environ.get('HF_TOKEN', ''))
    print(f'Pushed to https://huggingface.co/{HF_REPO}')

## 6. Smoke Test — Run Trained Model

In [ ]:
# Switch model to inference mode (disables dropout, fuses LoRA for speed).
FastLanguageModel.for_inference(model)

# Use the last collected observation as a test prompt.
if not positive:
    raise RuntimeError('No positive-reward steps to test with — run data collection first.')
test_step = positive[-1]
test_prompt = test_step.prompt

inputs = tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': OPTIMIZER_SYSTEM},
        {'role': 'user', 'content': test_prompt},
    ],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=1024,
    temperature=0.7,
    do_sample=True,
)

generated = tokenizer.decode(
    outputs[0][inputs.shape[1]:],
    skip_special_tokens=True,
)

print('=== Trained model output (first 1000 chars) ===')
print(generated[:1000])